# Regression — Ev tapşırığı

Multiple Linear Regression, Polynomial Regression və qiymətləndirmə metrikləri.

**Qeyd:** Orijinal `bal_10.csv`, `bal_20.csv`, `bal_50.csv` faylları göndərilmədiyindən `data/` qovluğundakı məlumatlar tapşırığı işlətmək üçün yaradılmış nümunə verilənlərdir. Real nəticələr üçün müəllimin CSV faylları ilə əvəz edin.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sutunlar = ["cicek_say", "temperatur", "rutubet"]
yeni_gun = pd.DataFrame([[180, 26, 55]], columns=sutunlar)
neticeler = []

## 1. Multiple Linear Regression və 2. Polynomial Regression

In [ ]:
for say in (10, 20, 50):
    ad = f"bal_{say}"
    df = pd.read_csv(Path("data") / f"{ad}.csv")
    x, y = df[sutunlar], df["bal_kq"]
    print(f"\n{ad}")
    display(df.head())

    linear = LinearRegression().fit(x, y)
    pred_linear = linear.predict(x)
    print("Multiple Linear | w:", dict(zip(sutunlar, linear.coef_.round(4))))
    print("Multiple Linear | b:", round(linear.intercept_, 4))
    print("Multiple Linear | yeni gün:", round(linear.predict(yeni_gun)[0], 3))
    neticeler.append([ad, "Multiple Linear", mean_absolute_error(y, pred_linear), mean_squared_error(y, pred_linear), r2_score(y, pred_linear)])

    poly = PolynomialFeatures(degree=2, include_bias=False)
    x2 = poly.fit_transform(x)
    model2 = LinearRegression().fit(x2, y)
    pred_poly = model2.predict(x2)
    print("Polynomial | əmsallar:", dict(zip(poly.get_feature_names_out(sutunlar), model2.coef_.round(4))))
    print("Polynomial | b:", round(model2.intercept_, 4))
    print("Polynomial | yeni gün:", round(model2.predict(poly.transform(yeni_gun))[0], 3))
    neticeler.append([ad, "Polynomial", mean_absolute_error(y, pred_poly), mean_squared_error(y, pred_poly), r2_score(y, pred_poly)])

## 3. Metrics cədvəli

In [ ]:
cedvel = pd.DataFrame(neticeler, columns=["Fayl", "Model", "MAE", "MSE", "R²"])
display(cedvel.round(4))

## 4. Qısa nəticə

Bu nümunə məlumatlarda modellərin müqayisəsi MAE və MSE-nin daha kiçik, R²-nin isə daha böyük olması ilə aparılır. Polynomial model eyni təlim məlumatlarında çox vaxt daha yaxşı nəticə verir, çünki əlavə kvadratik və qarşılıqlı təsir terminləri öyrənir. Lakin bu, yeni günlərdə mütləq daha dəqiq proqnoz verəcəyi demək deyil.

10, 20 və 50 sətirlik verilənlərdə məlumat artdıqca əmsalların qiymətləndirilməsi adətən daha sabit ola bilər. Xüsusilə 10 sətirlik nümunədə dərəcə-2 Polynomial model çox parametrli olduğundan overfitting riski yüksəkdir. Buradakı metriklər təlim məlumatları üzərində hesablanıb; modellərin ümumiləşdirmə qabiliyyətini yoxlamaq üçün ayrıca test məlumatları və ya cross-validation lazımdır.